In [ ]:
#判別システム全体
#前処理フェーズ，判別フェーズ

#前処理フェーズ：YOLO
#入力：画像フォルダ
#出力：前処理後の画像

#判別フェーズ
#入力：画像フォルダ
#出力：判別結果

In [ ]:
import os 
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import scipy.stats as stats
import seaborn as sns
import pandas as pd
from sklearn.metrics import roc_curve
import re
import math
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,precision_score, recall_score, f1_score
import joblib
import time
from ultralytics import YOLO

FlashAttention is not available on this device. Using scaled_dot_product_attention instead.


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
start = time.time()

In [3]:
data = "maesyori_img"

In [16]:
import os
import cv2
import numpy as np
from ultralytics import YOLO

# データフォルダの設定
data = "maesyori_img"

# 入力画像
input_file = "/home/data/maesyori_img/collage_1.jpg"

# 出力フォルダの作成
mask_output_folder = f"/home/data/{data}/mask/"
crop_output_folder = f"/home/data/{data}/crop/"
combined_output_folder = f"/home/data/{data}/combined/"  # 合成画像の出力フォルダ
os.makedirs(mask_output_folder, exist_ok=True)
os.makedirs(crop_output_folder, exist_ok=True)
os.makedirs(combined_output_folder, exist_ok=True)  # 合成フォルダを作成

# モデルの読み込み（物体検出用）
detection_model = YOLO('/home/YOLO/hukusuu_train/datasets/train7/weights/best.pt')
# モデルの読み込み（セグメンテーション用）
segmentation_model = YOLO("/home/YOLO/-327_seg/datasets/train2/weights/best.pt")

# 画像を処理
def process_image(image_path):
    results = detection_model.predict(image_path)  # 物体検出

    # 画像の読み込み
    orig_img = results[0].orig_img
    img_h, img_w, _ = orig_img.shape

    # 4×6のマスに分割
    rows, cols = 4, 6
    cell_h, cell_w = img_h // rows, img_w // cols

    # マス目ごとの bbox を格納するリスト
    cell_bboxes = [[[] for _ in range(cols)] for _ in range(rows)]

    # bbox を対応するマス目に格納
    if results[0].boxes is not None:
        for i, box in enumerate(results[0].boxes.xyxy):
            start_x, start_y, end_x, end_y = map(int, box)
            center_x, center_y = (start_x + end_x) // 2, (start_y + end_y) // 2
            
            row_idx = min(center_y // cell_h, rows - 1)
            col_idx = min(center_x // cell_w, cols - 1)
            
            cell_bboxes[row_idx][col_idx].append((start_x, start_y, end_x, end_y))

    # 出力フォルダを作成
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    image_output_folder = os.path.join(crop_output_folder, base_name)
    mask_output_folder_specific = os.path.join(mask_output_folder, base_name)
    combined_output_folder_specific = os.path.join(combined_output_folder, base_name)
    os.makedirs(image_output_folder, exist_ok=True)
    os.makedirs(mask_output_folder_specific, exist_ok=True)
    os.makedirs(combined_output_folder_specific, exist_ok=True)  # 合成フォルダを作成

    # マス目順に bbox を保存
    for row in range(rows):
        for col in range(cols):
            if not cell_bboxes[row][col]:  # bboxがない場合はスキップ
                continue
            for idx, (sx, sy, ex, ey) in enumerate(cell_bboxes[row][col]):
                clip_img = orig_img[sy:ey, sx:ex]
                clip_filename = os.path.join(image_output_folder, f"clip_{row+1}_{col+1}.jpg")
                cv2.imwrite(clip_filename, clip_img)

                # セグメンテーション用のマスク作成
                mask_results = segmentation_model.predict(clip_img)
                if mask_results and mask_results[0].masks is not None:
                    mask = mask_results[0].masks.data[0].cpu().numpy()  # CUDA -> CPU に転送して NumPy に変換
                    mask = (mask * 255).astype(np.uint8)  # 0-1 を 0-255 に変換

                    # マスク画像を切り抜き画像のサイズにリサイズ
                    mask_resized = cv2.resize(mask, (clip_img.shape[1], clip_img.shape[0]))

                    # マスク画像を保存
                    mask_filename = os.path.join(mask_output_folder_specific, f"mask_{row+1}_{col+1}.jpg")
                    cv2.imwrite(mask_filename, mask_resized)

                    # マスクと切り抜き画像を掛け合わせる（積を取る）
                    # clip_img_rgb = cv2.cvtColor(clip_img, cv2.COLOR_BGR2RGB)  # RGB形式に変換
                    mask_rgb = cv2.cvtColor(mask_resized, cv2.COLOR_GRAY2BGR)  # グレースケールのマスクをRGBに変換

                    # 切り抜き画像とリサイズしたマスク画像の積
                    combined_img = cv2.bitwise_and(clip_img, mask_rgb)  # 切り抜き画像とマスクの積

                    # 合成画像を保存
                    combined_filename = os.path.join(combined_output_folder_specific, f"combined_{row+1}_{col+1}.jpg")
                    cv2.imwrite(combined_filename, combined_img)

# 画像1枚のみ処理
process_image(input_file)



image 1/1 /home/data/maesyori_img/collage_1.jpg: 320x640 24 shiitake_bboxs, 14.0ms
Speed: 1.5ms preprocess, 14.0ms inference, 0.3ms postprocess per image at shape (1, 3, 320, 640)

0: 640x640 1 shiitake, 21.2ms
Speed: 0.8ms preprocess, 21.2ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 608x640 1 shiitake, 40.3ms
Speed: 1.9ms preprocess, 40.3ms inference, 0.6ms postprocess per image at shape (1, 3, 608, 640)

0: 640x576 1 shiitake, 19.1ms
Speed: 0.9ms preprocess, 19.1ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 576)

0: 640x640 1 shiitake, 33.3ms
Speed: 2.0ms preprocess, 33.3ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 shiitake, 20.7ms
Speed: 1.1ms preprocess, 20.7ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x608 1 shiitake, 20.3ms
Speed: 1.0ms preprocess, 20.3ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 608)

0: 640x608 1 shiitake, 19.6ms
Speed: 0.8ms preproc

In [ ]:
#判別フェーズ
#特徴量抽出
#形状
outputfile = f"/home/data/{data}/keijo_mse.csv"
inputfolder_lists = [
    f"/home/data/{data}/mask/collage_1/"
]

one_dimensional_data_dict = {}
evaluation_results = {}

for folder in inputfolder_lists:
    folder_name = os.path.basename(folder)
    image_paths = glob.glob(os.path.join(folder, '*.jpg'))
    
    # 画像ファイル数をカウント
    num_images = len(image_paths)
    print(f"Folder {folder_name} contains {num_images} images.")

    for img_path in image_paths:
        # 画像の読み込み
        mask = cv2.imread(img_path)

        # グレースケール画像に変換
        gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

        # 二値化
        _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # 輪郭を検出し、最大の輪郭を取得
        contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        max_contour = max(contours, key=cv2.contourArea)

        # 最小外接円を取得
        (x, y), radius = cv2.minEnclosingCircle(max_contour)
        radius = int(radius)

        # 重心を計算
        M = cv2.moments(max_contour)
        if M["m00"] != 0:
            cX = int(M["m10"] / M["m00"])
            cY = int(M["m01"] / M["m00"])
        else:
            cX, cY = 0, 0
        center = (cX, cY)

        # 極座標変換
        h, w = gray.shape
        flags = cv2.INTER_CUBIC + cv2.WARP_FILL_OUTLIERS + cv2.WARP_POLAR_LINEAR
        linear_polar = cv2.warpPolar(gray, (w, h), center, radius, flags)

        # 行ごとの黒ピクセル数をカウント
        black_pixel_count = np.sum(linear_polar == 0, axis=1)
        file_name = os.path.basename(img_path)
        one_dimensional_data_dict[file_name] = black_pixel_count

        # 真円の場合の理想データ（黒ピクセル数が0）
        y_pseudo = np.zeros_like(black_pixel_count)

        # 評価指標の計算
        mae = mean_absolute_error(y_pseudo, black_pixel_count)
        mse = mean_squared_error(y_pseudo, black_pixel_count)
        rmse = np.sqrt(mse)  # RMSEを計算

        evaluation_results[file_name] = {
            'MSE': mse, 
            'folder': folder_name
        }

# MSEの抽出
file_names = list(evaluation_results.keys())
mse_values = [metrics['MSE'] for metrics in evaluation_results.values()]
folders = [metrics['folder'] for metrics in evaluation_results.values()]
# DataFrame に変換
df_mse = pd.DataFrame(evaluation_results).T  # .T で転置して見やすくする
df_mse.reset_index(inplace=True)
df_mse.rename(columns={'index': 'filename'}, inplace=True)
# df_mse.rename(columns={'Folder': 'folder'}, inplace=True)
df_mse['filename'] = df_mse['filename'].astype(str)  # filenameを文字列型に
df_mse['MSE'] = df_mse['MSE'].astype(float)  # MSEを浮動小数型に
df_mse['folder'] = df_mse['folder'].astype(str)  # folderを文字列型に

input_folders = [
    f"/home/data/{data}/mask/collage_1/",
]

# 結果を保存するリスト
data_list = []

# データの収集
for input_folder in input_folders:
    for file in os.listdir(input_folder):
        if file.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.JPEG')):
            file_path = os.path.join(input_folder, file)
            image = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)

            # 二値化
            _, image = cv2.threshold(image, 128, 255, cv2.THRESH_BINARY)

            # 白ピクセルのカウント
            white_pixel_count = np.sum(image == 255)

            # データをリストに追加（folder情報は不要）
            data_list.append([file, white_pixel_count])

# DataFrame化（folder列を削除）
df_size = pd.DataFrame(data_list, columns=["filename", "size_count"])
df_size['filename'] = df_size['filename'].astype(str)
df_size['size_count'] = df_size['size_count'].astype(float)

#襞領域
def fmxy(absfxy, mxy):
    return np.where(absfxy > mxy, 1, 0)

def Min(a, b):
    return np.minimum(a, b)

def G12(theta1, theta2):
    condition1 = (theta2 - np.pi < theta1) & (theta1 < theta2) & (theta2 >= 0)
    result1 = np.abs(theta1 - theta2)
    condition2 = (-np.pi < theta1) & (theta1 < (theta2 - np.pi)) & (theta2 >= 0)
    result2 = theta2 - 2 * np.pi - theta1
    condition3 = (-np.pi < theta1) & (theta1 < (theta2 + np.pi)) & (theta2 < 0)
    result3 = np.abs(theta1 - theta2)
    condition4 = (theta2 + np.pi < theta1) & (theta1 < np.pi) & (theta2 < 0)
    result4 = theta1 - theta2 - 2 * np.pi
    result = np.where(condition1, result1, 
             np.where(condition2, result2, 
             np.where(condition3, result3, 
             np.where(condition4, result4, 0))))
    return result

def SquareSum(I, x, y, h, w, n):
    x1, y1 = x - n, y - n
    x2, y2 = x + n, y + n
    x1, x2 = max(x1, 0), min(x2, w - 2)
    y1, y2 = max(y1, 0), min(y2, h - 2)
    total = I[y2, x2] - I[y1, x2] - I[y2, x1] + I[y1, x1]
    return total

def sdis(Iruv, Imyv, x, y, h, w, n):
    Tr = SquareSum(Iruv, x, y, h, w, n)
    Tm = SquareSum(Imyv, x, y, h, w, n)
    return Tr / Tm

def extract_index(name):
    match = re.search(r'(\d+_\d+)', name)
    return match.group(1) if match else None


# パラメータ
n = 15
data_list = []

img_folders = [
    f"/home/data/{data}/combined/collage_1/",
]
mask_folders = [
    f"/home/data/{data}/mask/collage_1/",
]

for img_folder, mask_folder in zip(img_folders, mask_folders):
    folder_name = os.path.basename(img_folder)
    img_files = os.listdir(img_folder)
    mask_files = os.listdir(mask_folder)
    
    img_dict = {extract_index(f): f for f in img_files if extract_index(f)}
    mask_dict = {extract_index(f): f for f in mask_files if extract_index(f)}
    common_keys = sorted(set(img_dict.keys()) & set(mask_dict.keys()))

    for key in common_keys:
        img_file = img_dict[key]
        mask_file = mask_dict[key]
        img_path = os.path.join(img_folder, img_file)
        mask_path = os.path.join(mask_folder, mask_file)

        # 重心 (1)
        img = cv2.imread(img_path)
        mask_img = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        masked_img = cv2.imread(img_path)
        h, w = img.shape[:2]
        
        # 重心を計算
        x_sum, y_sum, count = 0, 0, 0
        for i in range(h):
            for j in range(w):
                if mask_img[i][j] == 255:
                    x_sum += j
                    y_sum += i
                    count += 1
        xc, yc = (x_sum / count, y_sum / count) if count > 0 else (0, 0)
        
        # fθ(x,y)(勾配の方向), |f(x,y)| (2)(3)
        image = cv2.cvtColor(masked_img, cv2.COLOR_BGR2GRAY)
        fdy, fdx = np.gradient(image)
        f0xy = np.arctan2(fdy, fdx)
        absfxy = np.uint8(np.sqrt(fdx**2 + fdy**2))
        
        # C0(x,y)：中心からのベクトルの角度
        height, width = image.shape
        C0xy = np.zeros((height, width))
        for y in range(height):
            for x in range(width):
                dx, dy = x - xc, y - yc
                C0xy[y, x] = np.arctan(dy / dx) if dx != 0 else 0
        
        # f(xy)の勾配ベクトルが中心から(x,y)へのベクトルへ垂直か評価する関数fdisxy(4)(5)
        fdisxy = Min(G12(C0xy + np.pi/2, f0xy)**2, G12(C0xy - np.pi/2, f0xy)**2)
        
        # mxy = |fxy|に対する2n+1×2n+1のメディアンフィルタリングの結果
        kernel_size = 2 * n + 1
        mxy = np.uint8(cv2.medianBlur(absfxy, kernel_size))
        
        # rdis (8)
        rdisxy = fmxy(absfxy, mxy) * fdisxy
        
        # Iruv, Imyv (9)(10)
        Iruv = cv2.integral(rdisxy)
        Imyv = cv2.integral(fmxy(absfxy, mxy).astype(np.uint8))
        
        # sdis計算
        sdisval = np.zeros((image.shape[0], image.shape[1]))
        for y in range(image.shape[0]):
            for x in range(image.shape[1]):
                sdisval[y, x] = sdis(Iruv, Imyv, x, y, h, w, n)
        sdisval = np.nan_to_num(sdisval, nan=0.0, posinf=0.0, neginf=0.0)
        
        # 閾値処理
        T = 0.2
        hxy = np.where(sdisval < T, 1, 0).astype(np.uint8)

        hxy2 = cv2.bitwise_and(hxy, hxy, mask=mask_img)
        
        # シイタケ領域のPixel数を計算
        count_mask = np.sum(mask_img == 255)
        count_hida = np.sum(hxy2 == 1)
        R = count_hida / count_mask if count_mask > 0 else 0
        data_list.append((img_file, R))

# CSV出力 & 結合
output_csv = f"/home/data/{data}/R_values.csv"
df_r = pd.DataFrame(data_list, columns=["filename", "R"])
df_r['filename'] = df_r['filename'].astype(str)
df_r['R'] = df_r['R'].astype(float)

df_mse['filename'] = df_mse['filename'].str.replace('mask_', '').str.replace('.jpg', '', regex=False)
df_size['filename'] = df_size['filename'].str.replace('mask_', '').str.replace('.jpg', '', regex=False)
df_r['filename'] = df_r['filename'].str.replace('combined_', '').str.replace('.jpg', '', regex=False)
df_merge = pd.merge(df_mse, df_size, on='filename', how='inner')
df_final = pd.merge(df_merge, df_r, on='filename', how='inner')
output_final_csv = f"/home/data/{data}/final_features.csv"
df_final.to_csv(output_final_csv, index=False)

Folder  contains 24 images.


/tmp/ipykernel_305189/3547252663.py:150: RuntimeWarning: invalid value encountered in scalar divide
  return Tr / Tm
/tmp/ipykernel_305189/3547252663.py:150: RuntimeWarning: divide by zero encountered in scalar divide
  return Tr / Tm


In [ ]:
# === モデルとスケーラーの読み込み ===
model_path = "svm_model.pkl"
scaler_path = "scaler.pkl"

svm_model = joblib.load(model_path)
scaler = joblib.load(scaler_path)

# === 新しいデータの読み込み ===
new_data_csv = f"/home/data/{data}/final_features.csv"  # 新しいデータ
df_new = pd.read_csv(new_data_csv)

# === 特徴量の抽出と標準化 ===
X_new = df_new[["MSE", "size_count", "R"]]  # 学習時と同じ特徴量を使用
X_new = scaler.transform(X_new)  # 標準化

# === 予測 ===
y_pred_new = svm_model.predict(X_new)

# 結果をDataFrameに追加
df_new["Predicted_Label"] = y_pred_new

# 予測結果の確認
print(df_new[["MSE", "size_count", "R", "Predicted_Label"]])

# CSVとして保存（オプション）
df_new.to_csv("/home/data/0203_energee_after/predicted_results.csv", index=False)


/usr/local/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


FileNotFoundError: [Errno 2] No such file or directory: '/home/data/maesyori_img/final_data.csv'

In [19]:
end = time.time()
print(f"Total time: {end - start:.2f} sec")

Total time: 85.72 sec
